# Домашнее задание к занятию «Функции потерь и оптимизация»

Выполнил: Ярослав Золотухин

In [131]:
%pip install numpy
%pip install sklearn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [132]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [133]:
iris = datasets.load_iris()
iris.data

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [134]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [135]:
X,y = iris.data, iris.target
X_filtered = X[iris.target != 2]
y_filtered = y[iris.target != 2]

# Логистическая регрессия

In [136]:
class LogisticRegression:
    
    def __init__(self, lm=0.01, N=1000, method="gradient_descent"):
        self.lm = lm
        self.N = N
        self.weights = None
        self.bias = None
        self.method = method
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        if(self.method == 'gradient_descent'): self.gradient_descent(X, y, n_samples)
        if(self.method == 'rmsprop'): self.rmsprop(X, y, n_samples, n_features)
        if(self.method == 'nadam'): self.nadam(X, y, n_samples, n_features)
        

    def gradient_descent(self, X, y, n_samples):       
        # Градиентный спуск
        for _ in range(self.N):
            z = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigma(z)
            
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            
            self.weights -= self.lm * dw
            self.bias -= self.lm * db

    def rmsprop(self, X, y, n_samples, n_features):
        v_dw = np.zeros(n_features)
        v_db = 0.0
        beta = 0.9 # один из экспертов в ответах на вопросы посоветовал взять такое значение

        for _ in range(self.N):
            # Стандартный расчет предсказаний и ошибок
            z = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigma(z)
            errors = y_predicted - y
            
            # Базовые градиенты
            dw = (1 / n_samples) * np.dot(X.T, errors)
            db = (1 / n_samples) * np.sum(errors)

            # Накапливаем квадраты градиентов
            v_dw = beta * v_dw + (1 - beta) * (dw ** 2)
            v_db = beta * v_db + (1 - beta) * (db ** 2)

            self.weights -= (self.lm / (np.sqrt(v_dw))) * dw
            self.bias -= (self.lm / (np.sqrt(v_db))) * db
            

    def nadam(self, X, y, n_samples, n_features):
        m_dw = np.zeros(n_features)
        v_dw = np.zeros(n_features)
        m_db = 0.0
        v_db = 0.0
        beta1 = 0.9
        beta2 = 0.99

        for t in range(1, self.N + 1):
            z = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigma(z)
            errors = y_predicted - y
            
            dw = (1 / n_samples) * np.dot(X.T, errors)
            db = (1 / n_samples) * np.sum(errors)

            # Обновление стандартных моментов Adam
            m_dw = beta1 * m_dw + (1 - beta1) * dw
            v_dw = beta2 * v_dw + (1 - beta2) * (dw ** 2)
            
            m_db = beta1 * m_db + (1 - beta1) * db
            v_db = beta2 * v_db + (1 - beta2) * (db ** 2)

            # Коррекция смещения (Bias correction)
            m_dw_hat = m_dw / (1.0 - beta1 ** t)
            v_dw_hat = v_dw / (1.0 - beta2 ** t)
            
            m_db_hat = m_db / (1.0 - beta1 ** t)
            v_db_hat = v_db / (1.0 - beta2 ** t)

            # Вычисление модифицированного импульса Нестерова
            nadam_dw = beta1 * m_dw_hat + ((1.0 - beta1) * dw) / (1.0 - beta1 ** t)
            nadam_db = beta1 * m_db_hat + ((1.0 - beta1) * db) / (1.0 - beta1 ** t)

            # Шаг оптимизации параметров
            self.weights -= (self.lm / (np.sqrt(v_dw_hat))) * nadam_dw
            self.bias -= (self.lm / (np.sqrt(v_db_hat))) * nadam_db


    def sigma(self, x):
        return 1 / (1 + np.exp(-x))
    
    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        y_predicted = self.sigma(z)
        return [1 if i > 0.5 else 0 for i in y_predicted]

In [137]:
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.3, random_state=42)

## Метод градиентного спуска

In [138]:
model_grad = LogisticRegression()

In [139]:
model_grad.fit(X_train, y_train)

In [140]:
model_grad.weights

array([-0.26332256, -1.04600252,  1.60171982,  0.69933897])

In [141]:
y_pred_grad = model_grad.predict(X_test)

In [142]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_grad):.2f}")

Accuracy: 1.00


## Метод RMSprop

In [143]:
model_rmsprop = LogisticRegression(method='rmsprop')

In [144]:
model_rmsprop.fit(X_train, y_train)

In [145]:
model_rmsprop.weights

array([ 0.5302178 , -8.18304247,  8.75551162,  8.79122114])

In [146]:
y_pred_rmsprop = model_rmsprop.predict(X_test)

In [147]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_grad):.2f}")

Accuracy: 1.00


## NADAM

In [148]:
model_nadam = LogisticRegression(method='nadam')

In [149]:
model_nadam.fit(X_train, y_train)

In [150]:
model_nadam.weights

array([ 0.25596876, -3.3990178 ,  3.52902533,  3.60533329])

In [151]:
y_pred_nadam = model_nadam.predict(X_test)

In [152]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_grad):.2f}")

Accuracy: 1.00


Вывод: Получается точность 100% у всех методов, скорость методов тоже очень быстрая (в пределах нуля). 
<br>
Либо это связано с небольшим количеством данных, либо я произвожу некорректный рассчет.